In [ ]:
!pip install --upgrade "mlflow>=3.1"
!pip install pandas
!pip install scikit-learn
!pip install boto3
!pip install python-dotenv

In [ ]:
from datetime import datetime
import pandas as pd
import s3fs
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient
from mlflow.models import infer_signature
import json
import shutil
from dotenv import load_dotenv

/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_name" in PromptModelConfig has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


In [3]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

In [ ]:
# =======================================================
# 🚨 CONFIGURACIÓN BASE (El equipo líder llena esto una vez)
# =======================================================

AUTOR_EXPERIMENTO = "Daniel Varela"

# 1. Los ganadores de las fases anteriores
WINNING_SPACY_FOLDER = "Exp04_Solo_Elongacion"
WINNING_VECTORIZER = "TFIDF"
WINNING_BIGRAMS = True

# ---------- Rutas y MLflow ----------
S3_BUCKET = os.getenv("S3_BUCKET_NAME")
TRAIN_PATH = f"{S3_BUCKET}/data/processed/Spacy/{WINNING_SPACY_FOLDER}/train_parquet"
DEST_BUCKET = f"{S3_BUCKET}/data/encode/Modelos"
DATASET_VERSION = WINNING_SPACY_FOLDER

S3_BUCKET_MLFLOW = os.getenv("S3_BUCKET_MLFLOW")
EC2_HOST = os.getenv("EC2_HOST")
MLFLOW_TRACKING_URI = f"{EC2_HOST}"
MLFLOW_ARTIFACT_BUCKET = f"{S3_BUCKET_MLFLOW}"
EXPERIMENT_NAME = "Sentimientos800"
REGISTERED_MODEL_NAME = "Sentimientos800"
TEAM = "NPL"

TEXT_COLUMN = "ablation_text"
TARGET_COLUMN = "label"
MAX_FEATURES = 10000

In [30]:
# =======================================================
#  DESCOMENTA SOLO EL BLOQUE DEL MODELO QUE TE TOCÓ
# =======================================================

# --- OPCIÓN 2: Random Forest (Modelo de Árboles) ---
from sklearn.ensemble import RandomForestClassifier
MODELO_ELEGIDO = "RandomForest"
clf = RandomForestClassifier(n_estimators=100, max_depth=40, n_jobs=-1, random_state=42 )

# --- OPCIÓN 3: Red Neuronal Densa (MLP) ---
# from sklearn.neural_network import MLPClassifier
# MODELO_ELEGIDO = "MLP"
# clf = MLPClassifier(hidden_layer_sizes=(100,), max_iter=20, early_stopping=True, random_state=42)

print(f"Modelo seleccionado y listo para entrenar: {MODELO_ELEGIDO}")

Modelo seleccionado y listo para entrenar: RandomForest


In [31]:
fs = s3fs.S3FileSystem()

def load_parquet_from_s3(prefix: str) -> pd.DataFrame:
    files = fs.ls(prefix)
    df_list = [pd.read_parquet(f"s3://{file}", filesystem=fs) for file in files]
    return pd.concat(df_list, ignore_index=True)

print(f"Cargando dataset: {WINNING_SPACY_FOLDER}...")
full_train_df = load_parquet_from_s3(TRAIN_PATH)

print("Columnas disponibles:", full_train_df.columns.tolist())

Cargando dataset: Exp04_Solo_Elongacion...
Columnas disponibles: ['text', 'label', 'clean_text', 'ablation_text']


In [32]:
X = full_train_df[TEXT_COLUMN].fillna("") 
y = full_train_df[TARGET_COLUMN]

print("Dividiendo en Entrenamiento (80%) y Validación (20%)...")
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

Dividiendo en Entrenamiento (80%) y Validación (20%)...


In [33]:
# 1. Configurar el vectorizador de la Fase B
ngram_range = (1, 2) if WINNING_BIGRAMS else (1, 1)

if WINNING_VECTORIZER == "TFIDF":
    vectorizer = TfidfVectorizer(max_features=MAX_FEATURES, ngram_range=ngram_range)
else:
    vectorizer = CountVectorizer(max_features=MAX_FEATURES, ngram_range=ngram_range)

# 2. Armar el Pipeline con el modelo descomentado
pipeline = Pipeline([
    ("vectorizer", vectorizer),
    ("model", clf)
])

print(f"Pipeline ensamblado: {WINNING_VECTORIZER} + {MODELO_ELEGIDO}")

Pipeline ensamblado: TFIDF + RandomForest


In [34]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

if experiment is None:
    artifact_location = f"{MLFLOW_ARTIFACT_BUCKET}/{EXPERIMENT_NAME}/"
    print(f"Creando experimento '{EXPERIMENT_NAME}' con artefactos en {artifact_location}")
    mlflow.create_experiment(name=EXPERIMENT_NAME, artifact_location=artifact_location)
else:
    print(f"Utilizando experimento existente '{EXPERIMENT_NAME}'")

mlflow.set_experiment(EXPERIMENT_NAME)

client = MlflowClient()

Utilizando experimento existente 'Sentimientos800'


In [35]:
run_name = f"Modelo_{MODELO_ELEGIDO}"

with mlflow.start_run(run_name=run_name):

    mlflow.set_tag("Autor", AUTOR_EXPERIMENTO)
    mlflow.set_tag("dataset_version", DATASET_VERSION)
    mlflow.set_tag("model_type", MODELO_ELEGIDO)
    mlflow.set_tag("team", TEAM)

    print(f"Entrenando {MODELO_ELEGIDO} (Por favor espera, modelos complejos tardan más)...")
    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_val)
    acc = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred, average="macro")

    print(f"Resultados -> Accuracy: {acc:.4f} | F1 Macro: {f1:.4f}")

    mlflow.log_params({
        "algoritmo": MODELO_ELEGIDO,
        "vectorizer_base": WINNING_VECTORIZER,
        "use_bigrams": WINNING_BIGRAMS,
        "spacy_base": WINNING_SPACY_FOLDER
    })

    mlflow.log_metrics({"accuracy": acc, "f1_macro": f1})

    signature = infer_signature(X_train, pipeline.predict(X_train))

    mlflow.sklearn.log_model(
        pipeline,
        artifact_path="ChampionModel",
        signature=signature,
        input_example=X_train.iloc[:3].to_frame(),
        registered_model_name=REGISTERED_MODEL_NAME
    )

print("Run guardado en MLflow.")

Entrenando RandomForest (Por favor espera, modelos complejos tardan más)...
Resultados -> Accuracy: 0.7465 | F1 Macro: 0.7461


2026/03/06 23:26:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/06 23:26:29 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'Sentimientos800' already exists. Creating a new version of this model...
2026/03/06 23:26:34 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Sentimientos800, version 18
Created version '18' of model 'Sentimientos800'.


🏃 View run Modelo_RandomForest at: http://ec2-52-21-111-46.compute-1.amazonaws.com:5000/#/experiments/11/runs/9cf085f946254529906e6bbb1dd984c5
🧪 View experiment at: http://ec2-52-21-111-46.compute-1.amazonaws.com:5000/#/experiments/11
Run guardado en MLflow.


In [36]:
local_tmp = "tmp_champion_model"

model_card = {
    "model_name": run_name,
    "description": f"Campeón Final. Algoritmo: {MODELO_ELEGIDO}. Vectorizador: {WINNING_VECTORIZER}. NLP: {WINNING_SPACY_FOLDER}.",
    "author": AUTOR_EXPERIMENTO,
    "dataset_version": DATASET_VERSION,
    "training_data_path": TRAIN_PATH,
    "version": str(datetime.now()),
    "performance": {"validation_accuracy": round(acc, 4), "validation_f1_macro": round(f1, 4)},
    "intended_use": "Inferencia productiva de clasificación de sentimientos."
}

versions = client.search_model_versions(f"name='{REGISTERED_MODEL_NAME}'")
latest_version = max(int(v.version) for v in versions)

try:
    champion_info = client.get_model_version_by_alias(REGISTERED_MODEL_NAME, "champion")
    champion_run = client.get_run(champion_info.run_id)
    champion_f1 = champion_run.data.metrics.get("f1_macro", 0)
    print(f"Campeón actual reinante: v{champion_info.version} | F1={champion_f1:.4f}")
except Exception:
    champion_f1 = -1
    print("Aún no hay campeón en esta Fase.")

print(f"F1 de tu modelo ({MODELO_ELEGIDO}): {f1:.4f}")

if f1 > champion_f1:
    print("¡TENEMOS UN NUEVO CAMPEÓN! Iniciando pase a producción...")
    client.set_registered_model_alias(name=REGISTERED_MODEL_NAME, alias="champion", version=latest_version)
    client.set_model_version_tag(name=REGISTERED_MODEL_NAME, version=latest_version, key="status", value="champion")

    with fs.open(f"{DEST_BUCKET}/model_card.json", "w") as f:
        f.write(json.dumps(model_card, indent=4))
    
    mlflow.artifacts.download_artifacts(artifact_uri=f"models:/{REGISTERED_MODEL_NAME}@champion", dst_path=local_tmp)
    fs.put(local_tmp, DEST_BUCKET, recursive=True)
    shutil.rmtree(local_tmp)
    print("¡ÉXITO! Modelo copiado a la capa Encode. La Lambda lo está enviando a Producción.")
else:
    print("Tu modelo no superó al Campeón actual. No se ha modificado Producción.")

Campeón actual reinante: v13 | F1=0.7981
F1 de tu modelo (RandomForest): 0.7461
Tu modelo no superó al Campeón actual. No se ha modificado Producción.
